# F_hash_cc_vs_dl — HASH Single-Event FMs: CC-SVD vs DL polarities

Runs HASH (`hash_driver3`) for 6 configurations:
- **Y1**: CC-SVD polarities, DL-all, DL ≥ 0.8 conf
- **Y2**: CC-SVD polarities, DL-all, DL ≥ 0.8 conf

All runs use `velmod_axial1D.dat` only. Minimum 8 polarities per event.
Kagan angles computed between CC-SVD FMs and both DL FM sets.


In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import h5py, subprocess, csv
from pathlib import Path
from datetime import datetime

DATA     = Path("/Users/mczhang/Documents/GitHub/FM7/02-data")
HASH_DIR = Path("/Users/mczhang/Documents/GitHub/FM7/01-scripts/HASH")
OUT_E    = DATA / "E_Po_py"
OUT_F    = DATA / "G_FM"; OUT_F.mkdir(exist_ok=True)

NPOLMIN  = 8
DL_CONF_MIN = 0.8
EH, EZ   = 0.3, 0.2
MAG      = 1.0
CHAN     = "HHZ"
NET      = "OO"
ONSET    = "I"

# Station metadata — lat, lon, elev(m)
STATION_META = {
    "AS1": (45.93356, -129.99920,  60),
    "AS2": (45.93377, -130.01410,  44),
    "CC1": (45.95468, -130.00890,  61),
    "EC1": (45.94958, -129.97970,  77),
    "EC2": (45.93967, -129.97380,  70),
    "EC3": (45.93607, -129.97850,  73),
    "ID1": (45.92573, -129.97800,  62),
    "01A": (46.01933, -130.00538,   0), "02A": (46.00063, -130.02926,  34),
    "03A": (45.99292, -129.99278, 102), "04A": (45.98874, -130.04840,  53),
    "05A": (45.97439, -129.97797,  73), "06A": (45.98294, -130.01413,   9),
    "07A": (45.96907, -130.02985,   8), "08A": (45.96415, -130.00450,  46),
    "09A": (45.97084, -130.06239, 103), "10A": (45.96034, -129.94999,  20),
    "11A": (45.94959, -130.03938, 158), "12A": (45.91537, -130.02178,  73),
    "13A": (45.90782, -129.97156,  24), "14A": (45.89997, -130.00757,   1),
    "15A": (45.91972, -129.93942,   0),
    "01B": (46.01935, -130.00526,   2), "02B": (46.00111, -130.02930,  32),
    "03B": (45.99250, -129.99397, 107), "04B": (45.98834, -130.04896,  55),
    "05B": (45.97327, -129.97728,  74), "06B": (45.98178, -130.01423,  12),
    "07B": (45.96950, -130.02963,  11), "08B": (45.96336, -130.00441,  57),
    "09B": (45.97194, -130.05972, 111), "10B": (45.95867, -129.94982,  23),
    "11B": (45.94851, -130.03801, 160), "12B": (45.91473, -130.02173,  78),
    "13B": (45.90472, -129.97240,  20), "14B": (45.89913, -130.00612,   1),
}

OO_STAS  = ["AS1","AS2","CC1","EC1","EC2","EC3","ID1"]
Y1_STAS  = OO_STAS + [f"{i:02d}A" for i in range(1, 15)]
Y2_STAS  = OO_STAS + [f"{i:02d}B" for i in range(1, 15)]

def short_to_hash(s):
    return "A" + s  # AS1->AAS1, 01A->A01A

def deg_to_degmin(decimal_deg):
    d = abs(decimal_deg)
    deg = int(d)
    mins = 60.0 * (d - deg)
    return deg, mins

def load_loc_file(path):
    locs = {}
    with open(path) as f:
        for line in f:
            parts = line.split()
            if len(parts) < 11: continue
            try:
                yr, mo, da = int(parts[0]), int(parts[1]), int(parts[2])
                hh, mn    = int(parts[3]), int(parts[4])
                ss        = float(parts[5])
                lat, lon, depth = float(parts[6]), float(parts[7]), float(parts[8])
                eid = int(parts[10])
                sec_i = int(ss)
                usec  = int(round((ss - sec_i) * 1e6))
                locs[eid] = (datetime(yr, mo, da, hh, mn, sec_i, usec), lon, lat, depth)
            except (ValueError, IndexError):
                continue
    return locs

print("Config loaded.")
print(f"Y1 stations ({len(Y1_STAS)}): {Y1_STAS}")
print(f"Y2 stations ({len(Y2_STAS)}): {Y2_STAS}")


Config loaded.
Y1 stations (21): ['AS1', 'AS2', 'CC1', 'EC1', 'EC2', 'EC3', 'ID1', '01A', '02A', '03A', '04A', '05A', '06A', '07A', '08A', '09A', '10A', '11A', '12A', '13A', '14A']
Y2 stations (21): ['AS1', 'AS2', 'CC1', 'EC1', 'EC2', 'EC3', 'ID1', '01B', '02B', '03B', '04B', '05B', '06B', '07B', '08B', '09B', '10B', '11B', '12B', '13B', '14B']


In [2]:
# Load CC-SVD and DL polarities from E_Po_py npy files
# npy cols: [eid, u_val, svd_final, dl_pol, dl_conf, WTcatpol, top3_thr]

def load_polarities(year, stations):
    """
    Returns dict: eid -> {sta: (cc_pol, dl_pol, dl_conf)}
    cc_pol, dl_pol in {-1, 0, 1}; dl_conf in [0,1]
    """
    # Build eid -> sta -> (cc_pol, dl_pol, dl_conf)
    pol_by_eid = {}  # eid -> {sta: (cc_pol, dl_pol, dl_conf)}
    for sta in stations:
        npy_path = OUT_E / f"E_{year}_{sta}.npy"
        if not npy_path.exists():
            continue
        arr = np.load(npy_path)
        for row in arr:
            eid      = int(row[0])
            cc_pol   = int(row[2])   # svd_final
            dl_p     = int(row[3])   # dl_pol
            dl_c     = float(row[4]) # dl_conf
            if eid not in pol_by_eid:
                pol_by_eid[eid] = {}
            pol_by_eid[eid][sta] = (cc_pol, dl_p, dl_c)
    return pol_by_eid

# Load amplitude data: eid_idx_map per year, amp arrays per station
def load_amplitudes(year, stations):
    wave_h5 = DATA / "A_ID" / f"waves_{year}_ge8.h5"
    eid_idx = {}
    amp_data = {}
    if not wave_h5.exists():
        print(f"  WARNING: {wave_h5} not found, using dummy amplitudes")
        return eid_idx, amp_data
    with h5py.File(wave_h5, "r") as f:
        eids = f["event_ids"][:]
        eid_idx = {int(e): i for i, e in enumerate(eids)}
        for sta in stations:
            if f"amplitudes/{sta}" in f:
                amp_data[sta] = f[f"amplitudes/{sta}"][:]
    return eid_idx, amp_data

def get_amp(eid, sta, eid_idx, amp_data):
    """Returns (noise, p_amp, s_amp) or defaults."""
    if sta not in amp_data or eid not in eid_idx:
        return 1.0, 1.0, 1.0
    idx = eid_idx[eid]
    amp = amp_data[sta][idx]
    if np.any(np.isnan(amp)):
        return 1.0, 1.0, 1.0
    noise = max(float(amp[0]), 0.01)
    s_amp = max(float(amp[1]), 0.01)
    p_amp = max(float(amp[2]), 0.01)
    return noise, p_amp, s_amp

print("Loading polarities...")
pol_Y1 = load_polarities("Y1", Y1_STAS)
pol_Y2 = load_polarities("Y2", Y2_STAS)
print(f"Y1: {len(pol_Y1):,} events with polarity data")
print(f"Y2: {len(pol_Y2):,} events with polarity data")

print("Loading amplitudes...")
eid_idx_Y1, amp_Y1 = load_amplitudes("Y1", Y1_STAS)
eid_idx_Y2, amp_Y2 = load_amplitudes("Y2", Y2_STAS)
print(f"Y1 amplitude events: {len(eid_idx_Y1):,}")
print(f"Y2 amplitude events: {len(eid_idx_Y2):,}")


Loading polarities...


Y1: 3,077 events with polarity data
Y2: 20,530 events with polarity data
Loading amplitudes...
Y1 amplitude events: 3,105
Y2 amplitude events: 20,636


In [3]:
def write_hash_inputs(tag, year, stations, pol_by_eid, eid_idx, amp_data,
                       loc_by_id, pol_mode="cc"):
    """
    pol_mode: "cc"    -> use svd_final
              "dl_all" -> use dl_pol (any conf)
              "dl_hq"  -> use dl_pol where conf >= DL_CONF_MIN
    Returns list of (cluster_idx, eid) pairs written.
    """
    pha_path = HASH_DIR / f"phase_{tag}.dat"
    amp_path = HASH_DIR / f"amp_{tag}.dat"
    inp_path = HASH_DIR / f"hash.input_{tag}"
    out_base = HASH_DIR / f"hashout_{tag}"

    eid_map  = []
    cluster_idx = 0

    with open(pha_path, "w") as fpha, open(amp_path, "w") as famp:
        for eid in sorted(pol_by_eid):
            if eid not in loc_by_id:
                continue
            sta_dict = pol_by_eid[eid]
            dt, lon, lat, depth = loc_by_id[eid]
            depth = max(depth, 0.5)

            # Select polarities based on mode
            sta_data = []
            for sta in stations:
                if sta not in sta_dict:
                    continue
                cc_pol, dl_p, dl_c = sta_dict[sta]
                if pol_mode == "cc":
                    pol_val = cc_pol
                elif pol_mode == "dl_all":
                    pol_val = dl_p
                else:  # dl_hq
                    pol_val = dl_p if (not np.isnan(dl_c) and dl_c >= DL_CONF_MIN) else 0
                if pol_val == 0:
                    continue
                pol_char = "U" if pol_val > 0 else "D"
                sn = short_to_hash(sta)
                noise, p_amp, s_amp = get_amp(eid, sta, eid_idx, amp_data)
                sta_data.append((sn, pol_char, noise, p_amp, s_amp))

            if len(sta_data) < NPOLMIN:
                continue

            cluster_idx += 1
            eid_map.append((cluster_idx, eid))
            npicks = len(sta_data)

            yr = dt.year; mo = dt.month; da = dt.day
            hr = dt.hour; mn = dt.minute
            sec = dt.second + dt.microsecond / 1e6
            ilat, mlat = deg_to_degmin(lat)
            ilon, mlon = deg_to_degmin(abs(lon))
            ns = "N" if lat >= 0 else "S"
            ew = "W" if lon < 0 else "E"

            header = (
                f"{yr:4d}{mo:2d}{da:2d}{hr:2d}{mn:2d}{sec:5.2f}"
                f"{ilat:2d}{ns}{mlat:5.2f}{ilon:3d}{ew}{mlon:5.2f}"
                f"{depth:5.2f}{EH:6.2f}{EZ:6.2f}{MAG:5.2f}"
                f"{cluster_idx:>16} \n"
            )
            fpha.write(header)
            for (sn, pol_char, _, _, _) in sta_data:
                fpha.write(f"{sn:4s} {NET:2s}  {CHAN:3s} {ONSET} {pol_char}\n")
            fpha.write(f"{chr(32)*70}{cluster_idx} \n")

            famp.write(f"{cluster_idx}     {npicks}\n")
            for (sn, _, noise, p_amp, s_amp) in sta_data:
                famp.write(
                    f"{sn:4s} {CHAN:3s} {NET:2s}{chr(32)*17}"
                    f"{noise:10.3f} {noise:10.3f}"
                    f" {p_amp:10.3f} {s_amp:10.3f}\n"
                )

    # Write station file for this year (sorted alphabetically by HASH name)
    sta_sorted = sorted(stations, key=lambda s: short_to_hash(s))
    sta_path = HASH_DIR / f"station_{tag}.dat"
    with open(sta_path, "w") as f:
        for sta in sta_sorted:
            if sta not in STATION_META: continue
            lat_s, lon_s, elev_s = STATION_META[sta]
            sn = short_to_hash(sta)
            f.write(f"{sn:4s} {CHAN:3s}  {lat_s:.5f} {lon_s:10.5f} {elev_s:5d} {NET}\n")

    # Write control file (relative paths — hash_driver3 run from HASH_DIR)
    with open(inp_path, "w") as f:
        f.write(f"station_{tag}.dat\n")
        f.write("reverse.dat\n")
        f.write("Acor.dat\n")
        f.write(f"amp_{tag}.dat\n")
        f.write(f"phase_{tag}.dat\n")
        f.write(f"hashout_{tag}_1.dat\n")
        f.write(f"hashout_{tag}_2.dat\n")
        f.write(f"hashout_{tag}_3.dat\n")
        f.write(f"hashout_{tag}_4.dat\n")
        f.write("50000\n")  # nmax
        f.write("180\n")    # max azimuthal gap (180=unconstrained)
        f.write("5\n")      # dang
        f.write("30\n")     # nmc
        f.write("300\n")    # maxout
        f.write("2\n")      # ratmin
        f.write("0.10\n")   # badfrac
        f.write("2\n")      # 2 (HASH param)
        f.write("0\n")      # 0 (HASH param)
        f.write("0.30000\n")# qbadfrac
        f.write("25\n")     # delmax
        f.write("45\n")     # cangle
        f.write("0.750000\n")# prob_max
        f.write("1\n")      # number of velocity models
        f.write("velmod_axial1D.dat\n")

    # Save eid map
    map_path = OUT_F / f"hash_{tag}_eid_map.csv"
    with open(map_path, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["cluster_idx","event_id"]); w.writerows(eid_map)

    print(f"  {tag}: {cluster_idx:,} events written -> {pha_path.name}")
    return eid_map

# Write all 6 configurations
loc_Y1 = load_loc_file(DATA / "A_all" / "axial.Y1.mldd.loc.260317")
loc_Y2 = load_loc_file(DATA / "A_all" / "axial.Y2.mldd.loc.260317")
print(f"Y1 locations: {len(loc_Y1):,}  Y2 locations: {len(loc_Y2):,}")

print("Writing HASH inputs...")
eid_maps = {}
for tag, year, stations, pol_by_eid, eid_idx, amp_data, loc_by_id, mode in [
    ("Y1_cc",     "Y1", Y1_STAS, pol_Y1, eid_idx_Y1, amp_Y1, loc_Y1, "cc"),
    ("Y1_dl_all", "Y1", Y1_STAS, pol_Y1, eid_idx_Y1, amp_Y1, loc_Y1, "dl_all"),
    ("Y1_dl_hq",  "Y1", Y1_STAS, pol_Y1, eid_idx_Y1, amp_Y1, loc_Y1, "dl_hq"),
    ("Y2_cc",     "Y2", Y2_STAS, pol_Y2, eid_idx_Y2, amp_Y2, loc_Y2, "cc"),
    ("Y2_dl_all", "Y2", Y2_STAS, pol_Y2, eid_idx_Y2, amp_Y2, loc_Y2, "dl_all"),
    ("Y2_dl_hq",  "Y2", Y2_STAS, pol_Y2, eid_idx_Y2, amp_Y2, loc_Y2, "dl_hq"),
]:
    eid_maps[tag] = write_hash_inputs(tag, year, stations, pol_by_eid,
                                       eid_idx, amp_data, loc_by_id, mode)
print("Done writing HASH inputs.")


Y1 locations: 6,726  Y2 locations: 48,244
Writing HASH inputs...
  Y1_cc: 856 events written -> phase_Y1_cc.dat


  Y1_dl_all: 1,446 events written -> phase_Y1_dl_all.dat


  Y1_dl_hq: 677 events written -> phase_Y1_dl_hq.dat


  Y2_cc: 10,072 events written -> phase_Y2_cc.dat


  Y2_dl_all: 12,582 events written -> phase_Y2_dl_all.dat


  Y2_dl_hq: 6,457 events written -> phase_Y2_dl_hq.dat
Done writing HASH inputs.


## Cell 4: Run HASH

HASH runs are executed from the command line (too slow for a notebook cell with large Y2 datasets):

```bash
cd 01-scripts/HASH
for tag in Y1_cc Y1_dl_all Y1_dl_hq Y2_cc Y2_dl_all Y2_dl_hq; do
    ./hash_driver3 < hash.input_${tag} &
done
wait
```

The cell below checks the output files.


In [4]:
HASH_EXE = HASH_DIR / "hash_driver3"
tags = ["Y1_cc","Y1_dl_all","Y1_dl_hq","Y2_cc","Y2_dl_all","Y2_dl_hq"]

# Check which outputs exist; run any that are missing
for tag in tags:
    out1 = HASH_DIR / f"hashout_{tag}_1.dat"
    if out1.exists():
        n = sum(1 for _ in open(out1))
        print(f"  {tag}: {n} lines in {out1.name}")
    else:
        print(f"  {tag}: output not found — running now...")
        inp_path = HASH_DIR / f"hash.input_{tag}"
        with open(inp_path) as inp_file:
            result = subprocess.run(
                [str(HASH_EXE)], stdin=inp_file,
                capture_output=True, text=True, cwd=str(HASH_DIR))
        if result.returncode != 0:
            print(f"    ERROR: {result.stderr[:200]}")
        else:
            n = sum(1 for _ in open(out1)) if out1.exists() else 0
            print(f"    -> done: {n} lines")


  Y1_cc: 856 lines in hashout_Y1_cc_1.dat
  Y1_dl_all: 1446 lines in hashout_Y1_dl_all_1.dat
  Y1_dl_hq: 677 lines in hashout_Y1_dl_hq_1.dat
  Y2_cc: 10072 lines in hashout_Y2_cc_1.dat
  Y2_dl_all: 12582 lines in hashout_Y2_dl_all_1.dat
  Y2_dl_hq: 6457 lines in hashout_Y2_dl_hq_1.dat


In [5]:
# Load HASH output (hashout_*_1.dat) and merge with eid maps

def load_hash_out(path):
    """Parse hashout_*_1.dat -> DataFrame with cluster_idx, strike, dip, rake, quality."""
    rows = []
    with open(path) as fh:
        for line in fh:
            p = line.split()
            if len(p) < 31:
                continue
            try:
                rows.append(dict(
                    cluster_idx = int(p[0]),
                    strike      = float(p[21]),
                    dip         = float(p[22]),
                    rake        = float(p[23]),
                    quality     = p[28],
                    npolarity   = int(p[5]),
                    stdr        = float(p[26]),
                ))
            except (ValueError, IndexError):
                continue
    return pd.DataFrame(rows)

results = {}
for tag in ["Y1_cc","Y1_dl_all","Y1_dl_hq","Y2_cc","Y2_dl_all","Y2_dl_hq"]:
    out_path = HASH_DIR / f"hashout_{tag}_1.dat"
    if not out_path.exists():
        print(f"  WARNING: {out_path} not found")
        results[tag] = pd.DataFrame()
        continue
    df = load_hash_out(out_path)
    # Merge with eid map
    eid_df = pd.DataFrame(eid_maps[tag], columns=["cluster_idx","event_id"])
    df = df.merge(eid_df, on="cluster_idx", how="left")
    results[tag] = df
    print(f"  {tag}: {len(df):,} FMs loaded")

# Show quality breakdown
for tag, df in results.items():
    if len(df) == 0: continue
    qcounts = df["quality"].value_counts().to_dict()
    print(f"  {tag}: " + "  ".join(f"{q}={qcounts.get(q,0)}" for q in list("ABCD")))


  Y1_cc: 856 FMs loaded
  Y1_dl_all: 1,446 FMs loaded
  Y1_dl_hq: 677 FMs loaded
  Y2_cc: 10,072 FMs loaded
  Y2_dl_all: 12,582 FMs loaded
  Y2_dl_hq: 6,457 FMs loaded
  Y1_cc: A=45  B=220  C=241  D=350
  Y1_dl_all: A=156  B=407  C=383  D=500
  Y1_dl_hq: A=76  B=221  C=165  D=215
  Y2_cc: A=606  B=2028  C=1856  D=5582
  Y2_dl_all: A=1216  B=2755  C=2306  D=6305
  Y2_dl_hq: A=616  B=1552  C=1314  D=2975


In [6]:
# Kagan angle between two focal mechanisms
def _sdr2ptb(s, d, r):
    s, d, r = np.radians(s), np.radians(d), np.radians(r)
    n  = np.array([-np.sin(d)*np.sin(s),
                    np.sin(d)*np.cos(s),
                   -np.cos(d)])
    sl = np.array([ np.cos(r)*np.cos(s) + np.sin(r)*np.cos(d)*np.sin(s),
                    np.cos(r)*np.sin(s) - np.sin(r)*np.cos(d)*np.cos(s),
                   -np.sin(r)*np.sin(d)])
    T = (n + sl) / np.sqrt(2)
    P = (n - sl) / np.sqrt(2)
    B = np.cross(T, P)
    return P, T, B

def kagan_angle(s1, d1, r1, s2, d2, r2):
    P1,T1,B1 = _sdr2ptb(s1,d1,r1)
    P2,T2,B2 = _sdr2ptb(s2,d2,r2)
    R2 = np.column_stack([P2,T2,B2])
    min_ang = np.inf
    for sp,st,sb in [(1,1,1),(-1,-1,1),(1,-1,-1),(-1,1,-1)]:
        R1v = np.column_stack([sp*P1, st*T1, sb*B1])
        cos_a = np.clip((np.trace(R1v.T @ R2) - 1) / 2, -1, 1)
        min_ang = min(min_ang, np.degrees(np.arccos(cos_a)))
    return min_ang

kagan_vec = np.vectorize(kagan_angle)

# Merge CC vs DL-all and CC vs DL-hq for each year
comparisons = {}
for year in ["Y1","Y2"]:
    cc = results[f"{year}_cc"]
    dl_all = results[f"{year}_dl_all"]
    dl_hq  = results[f"{year}_dl_hq"]
    if len(cc) == 0: continue

    # CC vs DL-all
    if len(dl_all) > 0:
        m1 = cc[["event_id","strike","dip","rake","quality"]].merge(
            dl_all[["event_id","strike","dip","rake","quality"]].rename(
                columns={"strike":"h_s","dip":"h_d","rake":"h_r","quality":"h_q"}),
            on="event_id", how="inner")
        m1["kagan"] = kagan_vec(m1["strike"],m1["dip"],m1["rake"],
                                 m1["h_s"],m1["h_d"],m1["h_r"])
        comparisons[f"{year}_cc_vs_dlall"] = m1
        print(f"{year} CC vs DL-all: {len(m1):,} matched, median Kagan={m1.kagan.median():.1f} deg")

    # CC vs DL-hq
    if len(dl_hq) > 0:
        m2 = cc[["event_id","strike","dip","rake","quality"]].merge(
            dl_hq[["event_id","strike","dip","rake","quality"]].rename(
                columns={"strike":"h_s","dip":"h_d","rake":"h_r","quality":"h_q"}),
            on="event_id", how="inner")
        m2["kagan"] = kagan_vec(m2["strike"],m2["dip"],m2["rake"],
                                 m2["h_s"],m2["h_d"],m2["h_r"])
        comparisons[f"{year}_cc_vs_dlhq"] = m2
        print(f"{year} CC vs DL-hq: {len(m2):,} matched, median Kagan={m2.kagan.median():.1f} deg")

    # DL-all vs DL-hq
    if len(dl_all) > 0 and len(dl_hq) > 0:
        m3 = dl_all[["event_id","strike","dip","rake","quality"]].merge(
            dl_hq[["event_id","strike","dip","rake","quality"]].rename(
                columns={"strike":"h_s","dip":"h_d","rake":"h_r","quality":"h_q"}),
            on="event_id", how="inner")
        m3["kagan"] = kagan_vec(m3["strike"],m3["dip"],m3["rake"],
                                 m3["h_s"],m3["h_d"],m3["h_r"])
        comparisons[f"{year}_dlall_vs_dlhq"] = m3
        print(f"{year} DL-all vs DL-hq: {len(m3):,} matched, median Kagan={m3.kagan.median():.1f} deg")


Y1 CC vs DL-all: 856 matched, median Kagan=65.0 deg

Y1 CC vs DL-hq: 509 matched, median Kagan=66.4 deg


Y1 DL-all vs DL-hq: 677 matched, median Kagan=16.9 deg


Y2 CC vs DL-all: 10,072 matched, median Kagan=55.9 deg


Y2 CC vs DL-hq: 5,942 matched, median Kagan=54.7 deg


Y2 DL-all vs DL-hq: 6,457 matched, median Kagan=17.1 deg


In [7]:
QUAL_COLORS = {"A":"#d62728","B":"#ff7f0e","C":"#2ca02c","D":"#9467bd"}
QUAL_ORDER  = list("ABCD")
bins = np.arange(0, 91, 5)

comp_labels = {
    "cc_vs_dlall":  "CC-SVD vs DL-all",
    "cc_vs_dlhq":   "CC-SVD vs DL-hq (≥0.8)",
    "dlall_vs_dlhq":"DL-all vs DL-hq (≥0.8)",
}

for year in ["Y1","Y2"]:
    comp_keys = [k for k in ["cc_vs_dlall","cc_vs_dlhq","dlall_vs_dlhq"]
                 if f"{year}_{k}" in comparisons]
    if not comp_keys:
        print(f"No comparisons for {year}")
        continue

    fig, axes = plt.subplots(2, len(comp_keys),
                             figsize=(5*len(comp_keys), 9),
                             facecolor="white")
    if len(comp_keys) == 1:
        axes = axes.reshape(2,1)

    for col, ck in enumerate(comp_keys):
        merged = comparisons[f"{year}_{ck}"]
        label  = comp_labels.get(ck, ck)

        # Top: stacked histogram
        ax = axes[0, col]
        data_by_q = [merged[merged["quality"]==q]["kagan"].values for q in QUAL_ORDER]
        ax.hist(data_by_q, bins=bins, stacked=True,
                color=[QUAL_COLORS[q] for q in QUAL_ORDER],
                label=[f"{q} (n={len(d):,})" for q,d in zip(QUAL_ORDER,data_by_q)],
                edgecolor="none", alpha=0.9)
        med = merged["kagan"].median()
        ax.axvline(med, color="k", ls="--", lw=1.2, label=f"Median={med:.1f}°")
        ax.set_title(f"{year}: {label}\n(n={len(merged):,})", fontsize=10)
        ax.set_xlabel("Kagan angle (°)"); ax.set_ylabel("Count")
        ax.set_xlim(0, 90); ax.legend(fontsize=8, framealpha=0.9)
        ax.grid(True, alpha=0.3, lw=0.5)

        # Bottom: CDF
        ax2 = axes[1, col]
        for q in QUAL_ORDER:
            sub = merged[merged["quality"]==q]["kagan"]
            if len(sub) == 0: continue
            sk = np.sort(sub)
            cdf = np.arange(1, len(sk)+1) / len(sk)
            ax2.plot(sk, cdf*100, color=QUAL_COLORS[q], lw=1.5,
                     label=f"{q} (n={len(sub):,}, med={sub.median():.0f}°)")
        ax2.axvline(30, color="grey", ls=":", lw=1)
        ax2.set_xlabel("Kagan angle (°)"); ax2.set_ylabel("Cumulative %")
        ax2.set_title(f"CDF by CC quality")
        ax2.legend(fontsize=8, framealpha=0.9)
        ax2.set_xlim(0, 90); ax2.set_ylim(0, 100)
        ax2.grid(True, alpha=0.3, lw=0.5)

    fig.suptitle(f"Kagan angles — {year} HASH single-event FMs", fontsize=12, y=1.01)
    plt.tight_layout()
    out_png = OUT_F / f"kagan_hash_{year}_cc_vs_dl.png"
    fig.savefig(out_png, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved -> {out_png}")

# Summary table
print("\n=== Summary ===")
for ck, label in comp_labels.items():
    for year in ["Y1","Y2"]:
        key = f"{year}_{ck}"
        if key not in comparisons: continue
        df = comparisons[key]
        pct30 = (df["kagan"] <= 30).mean() * 100
        print(f"  {year} {label}: n={len(df):,}  median={df.kagan.median():.1f}°  %<=30°={pct30:.1f}%")


Saved -> /Users/mczhang/Documents/GitHub/FM7/02-data/G_FM/kagan_hash_Y1_cc_vs_dl.png


Saved -> /Users/mczhang/Documents/GitHub/FM7/02-data/G_FM/kagan_hash_Y2_cc_vs_dl.png

=== Summary ===
  Y1 CC-SVD vs DL-all: n=856  median=65.0°  %<=30°=24.2%
  Y2 CC-SVD vs DL-all: n=10,072  median=55.9°  %<=30°=28.0%
  Y1 CC-SVD vs DL-hq (≥0.8): n=509  median=66.4°  %<=30°=19.8%
  Y2 CC-SVD vs DL-hq (≥0.8): n=5,942  median=54.7°  %<=30°=29.6%
  Y1 DL-all vs DL-hq (≥0.8): n=677  median=16.9°  %<=30°=67.2%
  Y2 DL-all vs DL-hq (≥0.8): n=6,457  median=17.1°  %<=30°=68.7%
